# Подключение LLM

## Задание 1. Первый запрос к модели через `/api/generate`

In [33]:
import requests

OLLAMA_URL = "http://localhost:11434"
LLM_MODEL = "qwen2.5:1.5b"


def generate_answer(prompt: str) -> str:
    """Отправляет промпт в Ollama и возвращает сгенерированный ответ."""

    response = requests.post(
        f"{OLLAMA_URL}/api/generate",
        json={
            "model": LLM_MODEL,
            "prompt": prompt,
            "stream": False,
        },
        timeout=120,
    )
    response.raise_for_status()
    data = response.json()

    return data["response"].strip()


# Пробуем
answer = generate_answer(prompt="Что такое RAG в одном предложении?")
print(answer)


RAG stands for Re-ranking Architecture, which is a type of model architecture used in recommendation systems. It is a re-ranking mechanism that improves the ranking of items in a recommendation system. This architecture is designed to give more weight to recent interactions with a user, rather than their overall historical behavior, to better predict what the user might want to see next. This can be particularly useful in scenarios where the user's preferences are expected to change over time, such as in personalized news feeds or streaming recommendations.


## Задание 2. Построение RAG промпта

In [ ]:
PROMPT_TEMPLATE = """Ты ассистент по документации Ollama.
Отвечай на вопрос пользователя, опираясь только на приведённые ниже фрагменты документации.
Если ответа во фрагментах нет - честно скажи, что не знаешь.
Ответ давай на русском языке.
В конце ответа на отдельной строке укажи источник в формате: Источник: <имя файла>.

Фрагменты документации:
{context}

Вопрос пользователя: {question}

Ответ:"""


def build_prompt(question: str, retrieved: list[dict]) -> str:
    """Собирает промпт из системной инструкции, контекста и вопроса."""
    context_parts = []
    for r in retrieved:
        context_parts.append(f"[файл: {r['source']}]\n{r['text']}")
    context = "\n\n---\n\n".join(context_parts)
    return PROMPT_TEMPLATE.format(context=context, question=question)

In [ ]:
def chunk_text(text: str, max_chars: int = 800, min_chars: int = 50) -> list[str]:
    """Режет Markdown по заголовкам ##; длинные секции дробит по параграфам.

    Параметры:
        text: исходный Markdown-текст
        max_chars: максимальная длина одного чанка в символах
        min_chars: минимальная длина чанка - чанки короче выбрасываются

    Возвращает:
        Список текстовых чанков
    """
    lines = text.split("\n")
    sections, current = [], []
    for line in lines:
        if line.startswith("## ") and current:
            sections.append("\n".join(current).strip())
            current = [line]
        else:
            current.append(line)
    if current:
        sections.append("\n".join(current).strip())

    chunks = []
    for section in sections:
        if not section:
            continue
        if len(section) <= max_chars:
            chunks.append(section)
            continue
        # Длинную секцию дробим по двойным переносам (параграфам)
        buf = ""
        for paragraph in section.split("\n\n"):
            if len(buf) + len(paragraph) + 2 <= max_chars:
                buf = f"{buf}\n\n{paragraph}" if buf else paragraph
            else:
                if buf:
                    chunks.append(buf.strip())
                buf = paragraph
        if buf:
            chunks.append(buf.strip())

    return [c for c in chunks if len(c) >= min_chars]


from pathlib import Path

DOCS_DIR = Path("docs")

chunks = []
for path in sorted(DOCS_DIR.rglob("*")):
    if path.suffix.lower() in {".md", ".mdx"}:
        text = path.read_text(encoding="utf-8")
        for chunk in chunk_text(text):
            chunks.append({"text": chunk, "source": str(path)})

print(f"Всего чанков: {len(chunks)}")


import numpy as np
from sentence_transformers import SentenceTransformer

EMBED_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

print(f"Загружаем модель эмбеддингов: {EMBED_MODEL}")
embed_model = SentenceTransformer(EMBED_MODEL)

# Берём только тексты чанков (метаданные оставим в chunks как есть)
texts = [c["text"] for c in chunks]

# Считаем эмбеддинги: на выходе - матрица (N, 384)
chunk_embeddings = embed_model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True,
).astype(np.float32)

print(f"Размер матрицы эмбеддингов: {chunk_embeddings.shape}")

import faiss

dim = chunk_embeddings.shape[1]  # 384
index = faiss.IndexFlatIP(dim)
index.add(chunk_embeddings)

print(f"В индексе {index.ntotal} векторов размерности {dim}")